## Modelo BaseLine


### By:
Jhonner Acosta

### Date:
2026-08-20

### Description:

Requerimiento
Crear un modelo base para tenerlo como referencia para luego comparar al entrenar modelos de machine learning, el modelo base puede ser un modelo de heurísticas o un modelo dummy. Crear un nuevo branch de git (Usar Gitflow).

Tomar como ejemplo los pasos de: https://joserzapata.github.io/post/ciencia-datos-proyecto-python/5-baseline_model/

En este proceso se incluye :

Usar pipeline de procesamiento de datos creado en la tarea anterior
Dividir los datos en Train / Test
Evaluar con diferentes métricas según el problema que se esta solucionando
Usar validación cruzada (Cross validation) y obtener la media y la desviación estándar de la medida de las evoluciones realizadas.
Seleccionar la variable mas importante y justificar por que se selecciona.
Realizar el análisis de Learning curve
obtener las gráficas de escalabilidad con tiempo de entrenamiento y score
Interpretar los resultados
realizar análisis de los resultados
realizar conclusiones
definir recomendaciones para crear un modelo en base a los resultados del modelo base
Propuestas y ideas en base a os resultados obtenidos
Puede utilizar las librerías o herramientas que considere para resolver la tarea

Entregables
Notebook con la descripción y creación de los pipelines de scikit-learn, además de los resultados del modelo base.
Se debe realizar un Pull request para ingresar el notebook a la ramamain para esto debe tener mínimo 1 revisiones de otras personas del Curso y que pase los check del CI/CD.

## 👷 Creación del Baseline Model

In [ ]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    cross_val_score,
    learning_curve,
    train_test_split,
)
from sklearn.pipeline import Pipeline

_root = next(
    p for p in [Path().resolve(), *Path().resolve().parents] if (p / "pyproject.toml").exists()
)
DATA_DIR = _root / "data"
MODELS_DIR = _root / "models"
SEED = 42
TEST_SIZE = 0.2
CV_FOLDS = 5

In [ ]:
df = pd.read_parquet(DATA_DIR / "02_intermediate" / "corazon_type_fixed.parquet")
df = df.drop_duplicates()

TARGET = "disease"
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

# Cargar pipeline de feature engineering
full_pipeline = joblib.load(MODELS_DIR / "feature_pipeline.pkl")

print(f"Shape dataset: {df.shape}")
print(f"Distribución target:\n{y.value_counts()}")

In [ ]:
# Preparar X compatible con el pipeline
cat_cols = ["sex", "chest_pain", "fbs", "rest_ecg", "slope", "ca", "thal"]
num_cols = ["age", "rest_bp", "chol", "max_hr", "old_peak"]

X_clean = X.copy()
X_clean = X_clean.drop(columns=["exang"], errors="ignore")
for col in cat_cols:
    X_clean[col] = X_clean[col].astype(object).where(X_clean[col].notna(), other=np.nan)
for col in num_cols:
    X_clean[col] = pd.to_numeric(X_clean[col], errors="coerce").astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target:\n{y_train.value_counts()}")
print(f"Test target:\n{y_test.value_counts()}")

In [18]:
baseline_pipeline = Pipeline(
    [
        ("preprocessor", full_pipeline.named_steps["preprocessor"]),
        ("classifier", DummyClassifier(strategy="most_frequent", random_state=SEED)),
    ]
)

baseline_pipeline.fit(X_train, y_train)
y_pred = baseline_pipeline.predict(X_test)
y_proba = baseline_pipeline.predict_proba(X_test)[:, 1]

print("=== Métricas en Test ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_test, y_proba):.4f}")
print("\nReporte de clasificación:\n")
print(classification_report(y_test, y_pred, zero_division=0))

=== Métricas en Test ===
Accuracy:  0.5556
F1 Score:  0.0000
AUC-ROC:   0.5000

Reporte de clasificación:

              precision    recall  f1-score   support

           0       0.56      1.00      0.71        45
           1       0.00      0.00      0.00        36

    accuracy                           0.56        81
   macro avg       0.28      0.50      0.36        81
weighted avg       0.31      0.56      0.40        81



In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
ax.set_title("Matriz de Confusión — Baseline")
plt.tight_layout()
plt.savefig("confusion_matrix_baseline.png", dpi=72)
plt.show()

In [ ]:
scoring_metrics = ["accuracy", "f1", "roc_auc"]

for metric in scoring_metrics:
    scores = cross_val_score(baseline_pipeline, X_clean, y, cv=CV_FOLDS, scoring=metric)
    print(f"{metric:10s}: {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
X_num = X_clean[num_cols].copy()
X_num[TARGET] = y.values

correlations = X_num.corr()[TARGET].drop(TARGET).abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
correlations.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Correlación absoluta con el target (disease)")
ax.set_ylabel("Correlación absoluta")
ax.set_xlabel("Feature")
plt.tight_layout()
plt.savefig("feature_correlation.png", dpi=72)
plt.show()

print(f"\nVariable más correlacionada con disease: {correlations.idxmax()}")

In [ ]:
train_sizes, train_scores, test_scores, fit_times, score_times = learning_curve(
    baseline_pipeline,
    X_clean,
    y,
    cv=CV_FOLDS,
    scoring="accuracy",
    train_sizes=np.linspace(0.1, 1.0, 10),
    return_times=True,
    random_state=SEED,
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
test_mean = test_scores.mean(axis=1)
test_std = test_scores.std(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Learning curve
axes[0].plot(train_sizes, train_mean, "o-", label="Train")
axes[0].plot(train_sizes, test_mean, "o-", label="Validation")
axes[0].fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2)
axes[0].fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.2)
axes[0].set_title("Learning Curve")
axes[0].set_xlabel("Training examples")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

# Scalability: fit time
axes[1].plot(train_sizes, fit_times.mean(axis=1), "o-")
axes[1].set_title("Escalabilidad — Tiempo de entrenamiento")
axes[1].set_xlabel("Training examples")
axes[1].set_ylabel("Tiempo (s)")

# Score vs fit time
axes[2].plot(fit_times.mean(axis=1), test_mean, "o-")
axes[2].set_title("Score vs Tiempo de entrenamiento")
axes[2].set_xlabel("Tiempo (s)")
axes[2].set_ylabel("Accuracy")

plt.tight_layout()
plt.savefig("learning_curve_baseline.png", dpi=72)
plt.show()

## 📊 Analysis of Results and Conclusions 

Description of the results obtained and if there are conclusions that can be drawn from them.

The analysis of results must be related to the description of the task.

**Note:** An analysis of results does not necessarily lead to conclusions, but to ideas or proposals for future work


## 8. Interpretación de resultados

El modelo baseline (DummyClassifier con estrategia `most_frequent`) predice siempre la clase mayoritaria (sin enfermedad). Sus métricas representan el piso mínimo que cualquier modelo real debe superar:

- **Accuracy: 0.5544** — refleja solo el desbalance de clases, no capacidad predictiva real.
- **F1: 0.0000** — no detecta ningún caso positivo (enfermedad), inaceptable en contexto médico.
- **AUC-ROC: 0.5000** — equivalente a clasificación aleatoria.

La variable numérica más correlacionada con el target es **`max_hr`** (frecuencia cardíaca máxima), lo que sugiere que tiene poder discriminativo para futuros modelos.

## 9. Conclusiones y recomendaciones

- El baseline establece un umbral mínimo de accuracy ~55% que cualquier modelo debe superar ampliamente.
- Dado el contexto médico, la métrica prioritaria debe ser **F1** o **Recall** de la clase positiva (enfermedad), no accuracy.
- Se recomienda explorar modelos como **Logistic Regression**, **Random Forest** o **Gradient Boosting**, que puedan capturar relaciones no lineales.
- El desbalance de clases (55/45) es manejable; no requiere técnicas de oversampling agresivas.
- Usar el pipeline de preprocesamiento del task 4 como base para todos los modelos siguientes.
- Priorizar features con alta correlación con el target: `max_hr`, `old_peak`, `age`